# 02 · Build the `events` table

**Input:** every `data/clean/tm_events_priced_<date>.csv` from `tm_full_pull.ipynb` (one per pull).
**Output:** `data/clean/events.csv`, one row per show, ready to join to `artists` and `venues`.

What this notebook does:
1. Combines all pulls, keeping each show's latest price (and its first price, to see changes)
2. Drops placeholder rows and big-venue names (arenas, stadiums)
3. Fills in missing headliners by parsing the event name
4. Makes `artist_key`, the join key to the artists table
5. Adds date features (days to show, weekend, etc.) and the target `log_price_min`

No API calls, so it runs in seconds.

In [1]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 80)
CLEAN = Path("data/clean")

## 1. Load and combine every pull

In [2]:
files = sorted(CLEAN.glob("tm_events_priced_*.csv"))
assert files, "No tm_events_priced_*.csv found in data/clean. Run tm_full_pull.ipynb first."
print("Pulls found:", [f.name for f in files])

raw = pd.concat((pd.read_csv(f, low_memory=False) for f in files), ignore_index=True)
raw = raw.sort_values("pull_date")

# first price seen and number of pulls each show appeared in
first = raw.groupby("event_id").agg(price_min_first=("price_min", "first"),
                                    price_max_first=("price_max", "first"),
                                    first_pull=("pull_date", "first"),
                                    n_pulls=("pull_date", "nunique"))
ev = raw.drop_duplicates("event_id", keep="last").merge(first, on="event_id")
ev["price_min_changed"] = ev["n_pulls"].gt(1) & ev["price_min"].ne(ev["price_min_first"])
print(f"{len(raw)} rows across pulls -> {len(ev)} unique shows")
print(f"Shows seen in 2+ pulls: {(ev['n_pulls'] > 1).sum()} | min price changed: {ev['price_min_changed'].sum()}")

Pulls found: ['tm_events_priced_2026-10-19.csv', 'tm_events_priced_2026-11-02.csv']
11219 rows across pulls -> 6877 unique shows
Shows seen in 2+ pulls: 4342 | min price changed: 121


## 2. Drop placeholders and big venues
Scope is small and mid-size venues. Venue names with *arena, stadium* etc. are dropped here; amphitheaters are only flagged, since some are small. The venues notebook adds real capacity later.

In [3]:
DROP_NAME = re.compile(r"^\s*TBD\b|TBD/TBD|^\s*TBA\b", re.I)
BIG_VENUE = re.compile(r"\b(?:arena|stadium|coliseum|ballpark|speedway|dome|fairgrounds?)\b", re.I)
AMPHI = re.compile(r"amphitheat(?:er|re)", re.I)

n0 = len(ev)
ev = ev[~ev["name"].str.contains(DROP_NAME, na=False)]
print("Dropped placeholder names:", n0 - len(ev))

big = ev["venue"].str.contains(BIG_VENUE, na=False)
print("Dropped big-venue names:", big.sum())
display(ev.loc[big, ["venue", "city", "state"]].value_counts().head(10).to_frame("events"))
ev = ev[~big].copy()
ev["flag_amphitheater"] = ev["venue"].str.contains(AMPHI, na=False)
print("Amphitheaters kept but flagged:", ev["flag_amphitheater"].sum())

Dropped placeholder names: 136
Dropped big-venue names: 2


,,,events
venue,city,state,
Fred Smith Rodeo Arena,Okeechobee,FL,2


Amphitheaters kept but flagged: 31


## 3. Fill in missing headliners

When Ticketmaster has no attraction listed, the artist is usually at the start of the event name:

| Event name | Parsed headliner |
|---|---|
| `Lady Radiator w/ Helen 55 & Upon Your Dead Body` | Lady Radiator |
| `BROOKE ALEXX: Dream Girl Tour` | BROOKE ALEXX |
| `*SOLD OUT* Blake Whiten` | Blake Whiten |
| `Swim Team / Auburn / House Warmer` | Swim Team |

Theme nights (tributes, dance parties, brunches) are flagged, because they have no single artist to look up.

In [4]:
SEP = re.compile(
    r"\s+(?:w/|with|feat\.?|featuring|ft\.?|x|vs\.?|\+|\|)\s+"  # word separators
    r"|,|;"                                                            # lists
    r"|\s/\s?|\s?/\s"                                                # ' / ' (keeps AC/DC)
    r"|\s+[-–—]\s+"                                                   # ' - Tour Name'
    r"|:\s", re.I)
TRAILING = re.compile(
    r"\s+(?:album release|record release|ep release|release show|residency.*|live|"
    r"the .*tour|.*\btour\b.*|\d{4})$", re.I)
THEME = re.compile(
    r"tribute|dance party|\bparty\b|\bnight\b|brunch|happy hour|karaoke|open mic|jam session|"
    r"showcase|festival|fest\b|celebration|(?:mon|tues|wednes|thurs|fri|satur|sun)days?\b|"
    r"\bvs\.?\b|emo nite|drag|comedy|bingo", re.I)


def parse_headliner(name):
    if not isinstance(name, str):
        return None
    s = re.sub(r"\*[^*]*\*", " ", name)          # *SOLD OUT*
    s = re.sub(r"\([^)]*\)|\[[^\]]*\]", " ", s)  # (Tribute), [Late Show]
    s = s.replace("’", "'").strip()
    m = re.search(r"\b(presents?):?\s+(.+)", s, re.I)
    if m:
        before, word, after = s[:m.start()].strip(), m.group(1).lower(), m.group(2).strip()
        # 'X & Y present ...' or 'Artist Presents: "Album" Tour' -> artist is before;
        # 'Promoter presents: Artist' -> artist is after
        artist_first = word == "present" or after.startswith(('"', "“", "'")) or "tour" in after.lower()
        s = before if (artist_first and before) else after
    first = SEP.split(s, maxsplit=1)[0]
    first = TRAILING.sub("", first).strip(" -:,.'\"“”")
    first = re.sub(r"\s{2,}", " ", first)
    return first or None


def make_key(name):
    """Join key: no accents, lowercase, letters and digits only, no leading 'the'."""
    if not isinstance(name, str) or not name.strip():
        return None
    s = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    s = s.lower().replace("&", " and ")
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()
    s = re.sub(r"^the\s+", "", s)
    return s or None


ENSEMBLE = re.compile(r"\s+(trio|quartet|quintet|sextet|septet|octet|nonet|big band|orchestra|band|ensemble)$", re.I)

ev["headliner_parsed"] = ev["name"].map(parse_headliner)
ev["headliner_source"] = np.where(ev["headliner"].notna(), "ticketmaster", "parsed_from_name")
ev["headliner_clean"] = ev["headliner"].fillna(ev["headliner_parsed"])
ev["artist_key"] = ev["headliner_clean"].map(make_key)
# a second name for the artist lookup: 'Mike Ledonne Trio' -> 'Mike Ledonne'
ev["lookup_name_alt"] = ev["headliner_clean"].str.replace(ENSEMBLE, "", regex=True).str.strip()
ev["is_theme_night"] = ev["name"].str.contains(THEME, na=False) & ev["headliner"].isna()

print(ev["headliner_source"].value_counts().to_string())
print("Missing artist_key after parsing:", ev["artist_key"].isna().sum())
print("Theme nights flagged:", ev["is_theme_night"].sum())

headliner_source
parsed_from_name    3838
ticketmaster        2901
Missing artist_key after parsing: 4
Theme nights flagged: 776


In [5]:
# Eyeball the parser: are these the right artists? Adjust SEP / TRAILING above if not.
(ev.loc[ev["headliner_source"] == "parsed_from_name", ["name", "headliner_clean", "is_theme_night"]]
   .sample(25, random_state=0))

,name,headliner_clean,is_theme_night
2410,John Splithoff - 2027 Tour,John Splithoff,False
5572,Melissa Errico - My Romance: Embracing the American Songbook,Melissa Errico,False
1911,"Slow Joy, aWannabe",Slow Joy,False
3541,ELAN TROTMAN - 25 YEARS IN MUSIC,ELAN TROTMAN,False
1602,"Pink Talking Fish - A Fusion of Pink Floyd, Talking Heads and Phish",Pink Talking Fish,False
4240,The Birdland Latin Jazz Orchestra,The Birdland Latin Jazz Orchestra,False
890,Cool Club & the Lipker Sisters: Halloween Bash!,Cool Club & the Lipker Sisters,False
5275,The Birdland Latin Jazz Orchestra,The Birdland Latin Jazz Orchestra,False
106,Mutate presents Restricted,Restricted,False
753,Haggis X-1,Haggis X-1,False


## 4. Date, timing and price features

In [6]:
ev["date"] = pd.to_datetime(ev["date"])
ev["pull_date"] = pd.to_datetime(ev["pull_date"])
ev["onsale_date"] = pd.to_datetime(ev["onsale_date"], utc=True, errors="coerce").dt.tz_localize(None)

ev["days_to_show"] = (ev["date"] - ev["pull_date"]).dt.days
ev["days_since_onsale"] = (ev["pull_date"] - ev["onsale_date"]).dt.days
ev["weekday"] = ev["date"].dt.day_name()
ev["is_weekend"] = ev["date"].dt.dayofweek.isin([4, 5])          # Friday and Saturday nights
ev["month"] = ev["date"].dt.month
ev["show_hour"] = pd.to_datetime(ev["local_time"], format="%H:%M:%S", errors="coerce").dt.hour

ev["has_support"] = ev["n_acts"].fillna(0).gt(1)
ev["single_price"] = ev["price_min"].eq(ev["price_max"])
ev["price_spread"] = ev["price_max"] - ev["price_min"]
ev["log_price_min"] = np.log(ev["price_min"])                      # TARGET
ev["flag_low_price"] = ev["price_min"] < 5                          # $0.26 tickets etc.: check before modeling
print("Shows with min price under $5:", ev["flag_low_price"].sum())

# genre: fold tiny or empty groups into Other
ev["genre"] = ev["genre"].replace({"Undefined": np.nan, "Music": np.nan}).fillna("Other")
counts = ev["genre"].value_counts()
ev["genre_grouped"] = ev["genre"].where(ev["genre"].map(counts) >= 30, "Other")
print(ev["genre_grouped"].value_counts().to_string())

Shows with min price under $5: 55
genre_grouped
Jazz                1634
Rock                1308
Alternative          798
Other                617
Country              443
Dance/Electronic     336
Pop                  328
Folk                 238
R&B                  206
Hip-Hop/Rap          204
Metal                179
World                154
Religious            131
Blues                103
Reggae                60


## 5. Final checks and save

In [7]:
# Same show listed twice once headliners are filled (e.g. early + late listing)
dups = ev.duplicated(["artist_key", "venue_id", "date", "local_time"]) & ev["artist_key"].notna()
print("Duplicate shows dropped:", dups.sum())
ev = ev[~dups]

keep = ["event_id", "name", "artist_key", "headliner_clean", "lookup_name_alt", "headliner_source",
        "is_theme_night", "support_acts", "n_acts", "has_support",
        "venue_id", "venue", "city", "state", "postal_code", "lat", "lon", "flag_amphitheater",
        "date", "local_time", "weekday", "is_weekend", "month", "show_hour",
        "onsale_date", "days_to_show", "days_since_onsale",
        "genre", "genre_grouped", "subgenre",
        "price_min", "price_max", "price_spread", "single_price", "log_price_min", "flag_low_price",
        "price_min_first", "price_min_changed", "n_pulls", "first_pull", "pull_date", "url"]
events = ev[keep].sort_values(["date", "venue"]).reset_index(drop=True)
events.to_csv(CLEAN / "events.csv", index=False)

print(f"Saved {CLEAN / 'events.csv'}: {len(events)} shows")
print(f"  states {events['state'].nunique()} | venues {events['venue_id'].nunique()} | "
      f"artists {events['artist_key'].nunique()} | DMV {events['state'].isin(['DC','MD','VA']).sum()}")
print("\nTarget (price_min) summary:")
events["price_min"].describe(percentiles=[.05, .25, .5, .75, .95]).round(2)

Duplicate shows dropped: 6
Saved data\clean\events.csv: 6733 shows
  states 42 | venues 506 | artists 3863 | DMV 53

Target (price_min) summary:


count    6733.00
mean       32.51
std        20.35
min         0.26
5%         12.56
25%        21.83
50%        30.14
75%        41.89
95%        54.99
max       799.10
Name: price_min, dtype: float64

In [8]:
# Missing values per column (anything high needs a decision before modeling)
events.isna().mean().sort_values(ascending=False).head(10).round(3).to_frame("share_missing")

,share_missing
support_acts,0.942
subgenre,0.128
artist_key,0.001
lookup_name_alt,0.000
headliner_clean,0.000
name,0.000
headliner_source,0.000
is_theme_night,0.000
n_acts,0.000
event_id,0.000
